In [ ]:

from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser

In [ ]:
import os
HF_TOKEN=os.getenv("HF_TOKEN")
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
# print(HF_TOKEN)
# print(GOOGLE_API_KEY)


In [16]:
# Loading LLM

llm=ChatGoogleGenerativeAI(
        model="gemini-3.5-flash",
        google_api_key=GOOGLE_API_KEY,
    )

In [17]:
# creating custom prompt
prompt=PromptTemplate(template="""
Use the pieces of information provided in the context to answer user's question.
If you dont know the answer, just say that you dont know, dont try to make up an answer. 
Dont provide anything out of the given context

Context: {context}
Question: {question}

Start the answer directly. No small talk please.""" , input_variables=["context", "question"])


In [18]:
# load dataset
Db_Path="vectorstore\db_faiss"
embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

DB=FAISS.load_local(Db_Path,embedding_model,allow_dangerous_deserialization=True)

# Retriever
retriever=DB.as_retriever(search_kwargs={'k':3})


<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5664\3424160510.py:2: SyntaxWarning: invalid escape sequence '\d'
  Db_Path="vectorstore\db_faiss"
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2506.42it/s]


In [19]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

result = chain.invoke("What is self attention")
print(result)

Self-attention, sometimes called intra-attention, is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence.
